## Problem statement

Shortest path from HCMUT to the famous Ben Thanh Market

This notebook uses Yen's algo to find k-shortest path

In [35]:
import math
import heapq
import json

import osmnx as ox
import networkx as nx
import pandas as pd
import folium

In [36]:
# Format: (latitude, longitude)
HCMUT = (10.7722, 106.6579)
BEN_THANH = (10.7725, 106.6983)

# Bounding box between two point
margin = 0.015

south = min(HCMUT[0], BEN_THANH[0]) - margin
north = max(HCMUT[0], BEN_THANH[0]) + margin
west = min(HCMUT[1], BEN_THANH[1]) - margin
east = max(HCMUT[1], BEN_THANH[1]) + margin

# OSMnx v2 style bbox: (west, south, east, north)
bbox = (west, south, east, north)

In [37]:
# Download road graph from OpenStreetMap
G_multi = ox.graph_from_bbox(
    bbox,
    network_type="drive",
    simplify=True,
    retain_all=False
)

print("Original graph type:", type(G_multi))
print("Original MultiDiGraph nodes:", G_multi.number_of_nodes())
print("Original MultiDiGraph edges:", G_multi.number_of_edges())

Original graph type: <class 'networkx.classes.multidigraph.MultiDiGraph'>
Original MultiDiGraph nodes: 3902
Original MultiDiGraph edges: 9118


In [38]:
# Convert MultiDiGraph -> DiGraph

# MultiDiGraph can have many edges u -> v.
# We want to keep only the shortest edge between the same u -> v.
D = ox.convert.to_digraph(G_multi, weight="length")

print("\nAfter converting to DiGraph:")
print("DiGraph type:", type(D))
print("DiGraph nodes:", D.number_of_nodes())
print("DiGraph edges:", D.number_of_edges())


After converting to DiGraph:
DiGraph type: <class 'networkx.classes.digraph.DiGraph'>
DiGraph nodes: 3902
DiGraph edges: 9087


In [39]:
graph_dict = {}

for u in D.nodes:
    graph_dict[u] = []

    for v, data in D[u].items():
        distance = float(data.get("length", 1.0))
        graph_dict[u].append((v, distance))

print("\nAdjacency list sample:")
sample_nodes = list(graph_dict.keys())[:3]
for node in sample_nodes:
    print(node, "->", graph_dict[node][:5])



Adjacency list sample:
366368680 -> [(6637725827, 5.900371819741878), (6251877929, 84.54787730294001)]
366370087 -> [(4932516740, 28.32673278733573), (6169690260, 72.20389397203482)]
366370617 -> [(366433356, 74.59296472026847), (366469000, 53.63004444023326), (11998294099, 116.89245111656068)]


In [40]:
def dijksta(graph: dict, source, debug = False):

    if source not in graph:
        raise ValueError("Source node not in the graph")
    label = {}
    prev = {}
    pq = []

    for node, _ in graph.items():
        label[node] = math.inf
        prev[node] = -1

        if node == source:
            heapq.heappush(pq, (0, node))
        
    # Set the initial label for the source node
    label[source] = 0
    S = []
    
    while len(S) < len(graph):
        if not pq:
            break
        _, u = heapq.heappop(pq)
        if u in S:
            continue
        S.append(u)

        # Get the neighbors of u
        neighbors_u = graph[u]

        for neighbor, distance in neighbors_u:
            if (label[u] + distance) < label[neighbor]:
                label[neighbor] = label[u] + distance
                prev[neighbor] = u
                heapq.heappush(pq, (label[neighbor], neighbor))
        if debug:
            print("Choose:", u)
            print("Label:\n", label)
            print("pq:\n", pq)
    if debug:
        print("S:", S)
    return label, prev

def get_shortest_path_from_dijkstra(label, prev, dest):

    if dest not in label:
        raise ValueError("Destination is not in label") 
    if math.isinf(label[dest]):
        return None, math.inf
    shortest_path = [dest]
    prev_node = prev[dest]
    while prev_node != -1:
        shortest_path.append(prev_node)
        prev_node = prev[prev_node]
    return shortest_path[::-1], label[dest]


In [41]:
# Helper function
def copy_graph(graph):
    return{
        node: neighbors.copy() for node, neighbors in graph.items()
    }

def remove_edge(graph, u, v):
    if u not in graph:
        return 
    graph[u] ={
        (neighbor, weight)
        for neighbor, weight in graph[u]
        if neighbor != v
    }

def remove_node(graph, node):
    if node in graph:
        del graph[node]
    
    for u in graph:
        graph[u] = [
            (neighbor, weight)
            for neighbor, weight in graph[u]
            if neighbor != node
        ]

In [42]:
def get_edge_weight(graph, u, v):
    for neighbor, weight in graph[u]:
        if neighbor == v:
            return weight
    return math.inf

def get_path_distance(graph, path):
    total = 0

    for u, v in zip(path[:-1], path[1:]):
        weight = get_edge_weight(graph, u, v)
        if math.isinf(weight):
            return math.inf
        total += weight

    return total

def get_overlap_distance(graph, candidate_path, selected_paths):
    selected_edges = set()
    for path in selected_paths:
        selected_edges.update(zip(path[:-1], path[1:]))

    overlap = 0
    for u, v in zip(candidate_path[:-1], candidate_path[1:]):
        if (u, v) in selected_edges:
            overlap += get_edge_weight(graph, u, v)

    return overlap

In [43]:
def yen_algo(graph, source, target, k, overlap_penalty=0):
    """
    Yen's k-shortest paths algorithm.

    overlap_penalty > 0 favors more diverse routes by adding extra score
    for candidate edges that were already used by previously selected paths.
    The returned distance is still the real path distance, not the penalized score.
    """
    if k < 1:
        raise ValueError("k must be greater or equal to 1")
    first_label, first_prev = dijksta(graph, source)
    first_path, first_dist = get_shortest_path_from_dijkstra(first_label, first_prev, target)

    if first_path is None:
        return []
    
    A = []
    A.append((first_path, first_dist))
    B = []
    seen_candidates = set()
    
    for kth in range(1, k):
        previous_path = A[-1][0]

        # Consider every spur node on previous_path
        for i in range(len(previous_path) - 1):
            spur_node = previous_path[i]
            root_path = previous_path[:i+1]

            graph_copy = copy_graph(graph)

            # Remove edge
            for path, dist in A:
                if len(path) > i + 1 and path[:i+1] == root_path:
                    u = path[i]
                    v = path[i + 1]
                    remove_edge(graph_copy, u, v)
            # Remove node
            for root_node in root_path[:-1]:
                remove_node(graph_copy, root_node)

            if spur_node not in graph_copy:
                continue

            # Run dijkstra from spur_node to dest
            label, prev = dijksta(graph_copy, spur_node)
            spur_path, spur_dist = get_shortest_path_from_dijkstra(label, prev, target)

            if spur_path is None:
                continue

            total_path = root_path[:-1] + spur_path
            total_dist = get_path_distance(graph, total_path)

            path_tuple = tuple(total_path)

            if path_tuple not in seen_candidates:
                selected_paths = [path for path, dist in A]
                overlap_dist = get_overlap_distance(graph, total_path, selected_paths)
                candidate_score = total_dist + overlap_penalty * overlap_dist
                heapq.heappush(B, (candidate_score, total_dist, total_path))
                seen_candidates.add(path_tuple)

        if not B:
            break

        while B:
            candidate_score, candidate_dist, candidate_path = heapq.heappop(B)

            if all(candidate_path != path for path, dist in A):
                A.append((candidate_path, candidate_dist))
                break
    return A


In [44]:
source = ox.distance.nearest_nodes(
    D,
    X=HCMUT[1],    # longitude
    Y=HCMUT[0]     # latitude
)

target = ox.distance.nearest_nodes(
    D,
    X=BEN_THANH[1],
    Y=BEN_THANH[0]
)

print("\nSource node near HCMUT:", source)
print("Target node near Ben Thanh:", target)



Source node near HCMUT: 12817685128
Target node near Ben Thanh: 2899065691


In [45]:

label, prev = dijksta(graph_dict, source, debug=False)

In [46]:
route = get_shortest_path_from_dijkstra(label, prev, target)
route

([12817685128,
  4628048101,
  4628048100,
  11584778514,
  4655963176,
  2302073725,
  11932199876,
  4666695460,
  3088844994,
  4662998247,
  6013212713,
  6791509618,
  5073779377,
  6751290372,
  6791017603,
  9843908828,
  9843908827,
  366370938,
  366419772,
  5772606965,
  2690659780,
  4927867379,
  6728906454,
  6728906452,
  5352824737,
  366385591,
  3718387345,
  366377012,
  2930534049,
  5074798778,
  5063701495,
  4120151674,
  366403850,
  366464770,
  4875861250,
  5063701408,
  411918284,
  366470932,
  366434259,
  10046834524,
  10046834527,
  5778164496,
  10046800715,
  13623759833,
  2690659614,
  2690659695,
  2690659691,
  411921991,
  4646769645,
  411919418,
  411919383,
  5778245684,
  411919210,
  411926405,
  6794477957,
  11953547345,
  2393615144,
  2393615146,
  2393614152,
  2393614147,
  2393614146,
  411924874,
  4612970101,
  6786738904,
  4609310620,
  4603207418,
  4610625918,
  4610625919,
  4666207251,
  9864826744,
  4610625914,
  5500064107,

In [47]:
# Get k routes. Increase OVERLAP_PENALTY to prefer routes that share fewer edges.
OVERLAP_PENALTY = 3.0
routes = yen_algo(graph_dict, source, target, k=3, overlap_penalty=OVERLAP_PENALTY)
print(len(routes))

3


In [50]:
routes

[([12817685128,
   4628048101,
   4628048100,
   11584778514,
   4655963176,
   2302073725,
   11932199876,
   4666695460,
   3088844994,
   4662998247,
   6013212713,
   6791509618,
   5073779377,
   6751290372,
   6791017603,
   9843908828,
   9843908827,
   366370938,
   366419772,
   5772606965,
   2690659780,
   4927867379,
   6728906454,
   6728906452,
   5352824737,
   366385591,
   3718387345,
   366377012,
   2930534049,
   5074798778,
   5063701495,
   4120151674,
   366403850,
   366464770,
   4875861250,
   5063701408,
   411918284,
   366470932,
   366434259,
   10046834524,
   10046834527,
   5778164496,
   10046800715,
   13623759833,
   2690659614,
   2690659695,
   2690659691,
   411921991,
   4646769645,
   411919418,
   411919383,
   5778245684,
   411919210,
   411926405,
   6794477957,
   11953547345,
   2393615144,
   2393615146,
   2393614152,
   2393614147,
   2393614146,
   411924874,
   4612970101,
   6786738904,
   4609310620,
   4603207418,
   4610625918,
  

In [48]:
def get_route_latlon(G, route):
    route_latlon = []

    for u, v in zip(route[:-1], route[1:]):
        data = G.get_edge_data(u, v)

        if data is None:
            continue

        geometry = data.get("geometry")

        if geometry is not None:
            # Shapely geometry coords are (lon, lat)
            points = [(lat, lon) for lon, lat in geometry.coords]
        else:
            points = [
                (G.nodes[u]["y"], G.nodes[u]["x"]),
                (G.nodes[v]["y"], G.nodes[v]["x"])
            ]

        if len(route_latlon) > 0 and route_latlon[-1] == points[0]:
            route_latlon.extend(points[1:])
        else:
            route_latlon.extend(points)

    return route_latlon


In [49]:
# Visualize full graph + k Dijkstra routes using Folium

# Use the original MultiDiGraph for OSMnx's GeoDataFrame conversion.
# D is a DiGraph, so ox.graph_to_gdfs(D) raises: edges(keys=True) is unsupported.
nodes_gdf, edges_gdf = ox.graph_to_gdfs(G_multi)

center_lat = (HCMUT[0] + BEN_THANH[0]) / 2
center_lon = (HCMUT[1] + BEN_THANH[1]) / 2

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=14,
    tiles="OpenStreetMap"
)

# Draw all road edges
folium.GeoJson(
    edges_gdf[["geometry"]].to_json(),
    name="Road network",
    style_function=lambda x: {
        "color": "gray",
        "weight": 1,
        "opacity": 0.4
    }
).add_to(m)

# Draw all shortest paths found by the k-route function.
# If routes does not exist yet, fall back to the single route variable.
route_candidates = routes if "routes" in globals() else []
if not route_candidates and "route" in globals() and route is not None:
    route_candidates = [route]

route_styles = [
    {"color": "red", "weight": 14, "dash_array": None},
    {"color": "blue", "weight": 10, "dash_array": "12, 8"},
    {"color": "green", "weight": 6, "dash_array": "3, 7"},
    {"color": "purple", "weight": 5, "dash_array": "10, 6, 2, 6"},
    {"color": "orange", "weight": 4, "dash_array": "2, 6"},
]

routes_drawn = 0

for i, (route_nodes, route_distance) in enumerate(route_candidates, start=1):
    route_latlon = get_route_latlon(D, route_nodes)
    if len(route_latlon) < 2:
        continue

    style = route_styles[(i - 1) % len(route_styles)]
    color = style["color"]
    route_layer = folium.FeatureGroup(
        name=f"Route {i}: {route_distance:.0f} m",
        show=True
    )

    # White casing makes each route visible on top of the road network.
    folium.PolyLine(
        route_latlon,
        color="white",
        weight=style["weight"] + 4,
        opacity=0.75,
        tooltip=f"Route {i}: {route_distance:.2f} meters"
    ).add_to(route_layer)

    # Different widths and dash patterns keep almost-overlapping routes visible.
    folium.PolyLine(
        route_latlon,
        color=color,
        weight=style["weight"],
        opacity=0.9,
        dash_array=style["dash_array"],
        tooltip=f"Route {i}: {route_distance:.2f} meters"
    ).add_to(route_layer)

    mid_point = route_latlon[len(route_latlon) // 2]
    folium.Marker(
        location=mid_point,
        tooltip=f"Route {i}: {route_distance:.2f} meters",
        icon=folium.DivIcon(
            html=f"""
            <div style="
                background:{color};
                color:white;
                border:2px solid white;
                border-radius:999px;
                width:24px;
                height:24px;
                line-height:20px;
                text-align:center;
                font-weight:bold;
                box-shadow:0 1px 4px rgba(0,0,0,0.45);
            ">{i}</div>
            """
        )
    ).add_to(route_layer)

    route_layer.add_to(m)
    routes_drawn += 1

# Marker HCMUT
folium.Marker(
    location=HCMUT,
    popup="HCMUT",
    icon=folium.Icon(color="green", icon="play")
).add_to(m)

# Marker Ben Thanh
folium.Marker(
    location=BEN_THANH,
    popup="Ben Thanh Market",
    icon=folium.Icon(color="orange", icon="flag")
).add_to(m)

folium.LayerControl().add_to(m)

import os

os.makedirs("outputs", exist_ok=True)
output_path = "outputs/k_dijkstra_routes_yen.html"
m.save(output_path)

print("\nSaved map:")
print(f"- {output_path}")
print(f"Routes found: {len(route_candidates)}")
print(f"Routes drawn: {routes_drawn}")


Saved map:
- outputs/k_dijkstra_routes_yen.html
Routes drawn: 3
